In [6]:
import google.generativeai as genai

for model_info in genai.list_tuned_models():
    print(model_info.name)


DefaultCredentialsError: 
  No API_KEY or ADC found. Please either:
    - Set the `GOOGLE_API_KEY` environment variable.
    - Manually pass the key with `genai.configure(api_key=my_api_key)`.
    - Or set up Application Default Credentials, see https://ai.google.dev/gemini-api/docs/oauth for more information.

## Lab: Fine Tuning            
Prof. OUHMIDA
## T.A.F: 
<ol>
<li>Démarrer le programme </li>
<li>Analyser le code </li>
<li>Améliorer le programme </li>
</ol>


source: https://ai.google.dev/gemini-api/docs/model-tuning/tutorial?lang=python&hl=fr

pip install -U google-generativeai
pip install python-dotenv

## 1- Installer le package du SDK et configurer votre clé API

In [8]:
# Importez le package et configurez le service avec votre clé API
import os
from dotenv import load_dotenv
import google.generativeai as genai
load_dotenv()  # Load environment variables from .env
genai.configure(api_key=os.environ['GOOGLE_API_KEY'])

In [19]:
 model = genai.GenerativeModel('models/gemini-1.5-flash-001')
response = model.generate_content("Expliquer comment fonctionne Fine Tuning")
print(response.text)

NotFound: 404 models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.

## 2- Répertorier les tuned-models(modèles affinés)

In [13]:
import google.generativeai as genai

for model_info in genai.list_tuned_models():
    print(model_info.name)

## 3- Créer un Tuned-Model

In [14]:
import google.generativeai as genai
import logging

logging.basicConfig(level=logging.INFO)

base_model = "models/gemini-1.5-flash-001-tuning"
training_data = [
    {"text_input": "1", "output": "2"},
    {"text_input": "2", "output": "3"},
    {"text_input": "3", "output": "4"},
    {"text_input": "10", "output": "11"},
    {"text_input": "12", "output": "13"},
    {"text_input": "100", "output": "101"},
    {"text_input": "1000", "output": "1001"},
    {"text_input": "I", "output": "II"},
    {"text_input": "II", "output": "III"},
    {"text_input": "III", "output": "IV"},
    {"text_input": "IV", "output": "V"},
    {"text_input": "five", "output": "six"},
    {"text_input": "six", "output": "seven"},
    {"text_input": "seven", "output": "eight"},
    {"text_input": "eleven", "output": "twelve"},
    {"text_input": "thirteen", "output": "fourteen"},
    {"text_input": "nineteen", "output": "twenty"}
]

try:
    operation = genai.create_tuned_model(
        display_name="increment",
        source_model=base_model,
        epoch_count=10,
        batch_size=4,
        learning_rate=0.3,
        training_data=training_data,
    )


    for status in operation.wait_bar():
        logging.debug(f"Status object: {vars(status)}") # To inspect what info we have available
        if hasattr(status, 'done') and status.done:
            logging.info("Fine-tuning operation completed.")
            break

        if hasattr(status, 'error') and status.error:
            logging.error(f"Fine-tuning failed with error: {status.error}")
            break
        # You can add some logging here to track progress. It can be different for different operations
        if hasattr(status, 'progress'):
            logging.info(f"Progress: {status.progress}")


    result = operation.result()
    logging.info(f"Fine-tuning complete. Tuned model name: {result.name}")

    model = genai.GenerativeModel(model_name=result.name)
    result = model.generate_content("III")
    logging.info(f"Generated output: {result.text}")  # Expected: IV
    #Basic test to see if it gives us what we expect.
    if "IV" in result.text:
        logging.info("Sanity check passed, model is working.")
    else:
        logging.warning(f"Sanity check failed, model did not return expected result: {result.text}")


except Exception as e:
    logging.error(f"An error occurred: {e}")

ERROR:root:An error occurred: 400 models/gemini-1.5-flash-001-tuning is not found for CREATE TUNED MODEL at API version v1beta.


In [ ]:
import google.generativeai as genai

import time

base_model = "models/gemini-1.5-flash-001-tuning"
training_data = [
    {"text_input": "1", "output": "2"},
    {"text_input": "2", "output": "3"},
    {"text_input": "3", "output": "4"},
    {"text_input": "10", "output": "11"},
    {"text_input": "12", "output": "13"},
    {"text_input": "100", "output": "101"},
    {"text_input": "1000", "output": "1001"},
    {"text_input": "I", "output": "II"},
    {"text_input": "II", "output": "III"},
    {"text_input": "III", "output": "IV"},
    {"text_input": "IV", "output": "V"},
    {"text_input": "five", "output": "six"},
    {"text_input": "six", "output": "seven"},
    {"text_input": "seven", "output": "eight"},
    {"text_input": "eleven", "output": "twelve"},
    {"text_input": "thirteen", "output": "fourteen"},
    {"text_input": "nineteen", "output": "twenty"}
]

operation = genai.create_tuned_model(
    # You can use a tuned model here too. Set `source_model="tunedModels/..."`
    display_name="increment",
    source_model=base_model,
    epoch_count=10,
    batch_size=4,
    learning_rate=0.1,
    training_data=training_data,
)

for status in operation.wait_bar():
    time.sleep(10)

result = operation.result()
print(result)
# # You can plot the loss curve with:
# snapshots = pd.DataFrame(result.tuning_task.snapshots)
# sns.lineplot(data=snapshots, x='epoch', y='mean_loss')

model = genai.GenerativeModel(model_name=result.name)
result = model.generate_content("III")
print(result.text)  # IV

In [ ]:
import google.generativeai as genai

model = genai.GenerativeModel(model_name="tunedModels/increment-29dov0wxblr4")
result = model.generate_content("III")
print(result.text)  # "IV"

## Mettre à jour la description

genai.update_tuned_model('tunedModels/my-increment-model', {"description":"This is my model."})

## Supprimer le modèle

In [ ]:
genai.delete_tuned_model("tunedModels/my-increment-model")

In [ ]:
genai.delete_tuned_model("tunedModels/increment-29dov0wxblr4")

In [ ]:
for model_info in genai.list_tuned_models():
    print(model_info.name)